<div style="background: linear-gradient(135deg, #1a3a1a 0%, #2d5a27 50%, #1a3a1a 100%); padding: 40px 30px; border-radius: 12px; color: white; font-family: 'Segoe UI', sans-serif;">

<div style="display: flex; align-items: center; gap: 20px; margin-bottom: 30px;">
<div style="background: rgba(0,0,0,0.06); padding: 20px; border-radius: 8px;">
</div>
<h1 style="margin: 0; font-size: 1.8em; font-weight: 700; letter-spacing: 1px;">INFORME TÉCNICO </h1>
</div>
<h2 style="margin: 4px 0 0; font-size: 1.1em; font-weight: 400; opacity: 0.9;">Pipeline ETL · Base de Datos FIA · Agrícola HC</h2>
</div>
</div>

<hr style="border-color: rgba(255,255,255,0.3); margin: 20px 0;">

<table style="width: 100%; border-collapse: collapse; font-size: 0.95em;">
<tr><td style="padding: 6px 0; opacity: 0.8; width: 180px;">📋 Proyecto</td><td style="padding: 6px 0; font-weight: 600;">Chirimoyo FIA · Digitalización Agrícola – Agrícola HC</td></tr>
<tr><td style="padding: 6px 0; opacity: 0.8;">📁 Repositorio</td><td style="padding: 6px 0; font-weight: 600;"><a href="https://github.com/Marce-717/Chirimoyo" style="color: #a8d5a2;">Marce-717/Chirimoyo</a> · GitHub</td></tr>
<tr><td style="padding: 6px 0; opacity: 0.8;">👤 Responsable</td><td style="padding: 6px 0; font-weight: 600;">Marcelo Toro Miranda · Ing. Agrónomo MSc</td></tr>
<tr><td style="padding: 6px 0; opacity: 0.8;">⚙️ Ejecutor técnico</td><td style="padding: 6px 0; font-weight: 600;">Nicolás Cortés - Ing. Agrónomo</td></tr>
<tr><td style="padding: 6px 0; opacity: 0.8;">📅 Versión / Fecha</td><td style="padding: 6px 0; font-weight: 600;">v2.0 · Julio 2026</td></tr>
<tr><td style="padding: 6px 0; opacity: 0.8;">🏛️ Institución</td><td style="padding: 6px 0; font-weight: 600;">Universidad de Chile · Facultad de Ciencias Agronómicas</td></tr>
</table>

</div>

---

## 📑 Índice de Contenidos

| # | Sección | Descripción |
|---|---------|-------------|
| 1 | [Contexto y Objetivos](#1-contexto-y-objetivos) | Marco del proyecto FIA y alcance del entregable |
| 2 | [Fuente de Datos](#2-fuente-de-datos) | Descripción del archivo Excel origen |
| 3 | [Arquitectura de la Solución](#3-arquitectura-de-la-solución) | Diseño general del pipeline ETL |
| 4 | [Modelo Relacional](#4-modelo-relacional) | Diccionario de datos y esquema de la BD |
| 5 | [Infraestructura Tecnológica](#5-infraestructura-tecnológica) | Stack tecnológico y configuración |
| 6 | [Pipeline ETL · Detalle Técnico](#6-pipeline-etl--detalle-técnico) | Fases de extracción, transformación y carga |
| 7 | [Calidad de Datos](#7-calidad-de-datos) | Reglas de negocio y reporte de validación |
| 8 | [Resultados de la Ingesta](#8-resultados-de-la-ingesta) | Métricas y estadísticas post-carga |
| 9 | [Archivos Entregables](#9-archivos-entregables) | Inventario de archivos del proyecto |

---

<a id="1-contexto-y-objetivos"></a>
## 1. Contexto y Objetivos

### 1.1 Marco del Proyecto FIA

El presente informe documenta el proceso de **diseño, implementación y carga** de una base de datos relacional para el Proyecto FIA de investigación en cultivo de chirimoyos de Agrícola HC. Este proyecto es parte de una iniciativa de **digitalización agrícola** que busca centralizar, estandarizar y hacer analíticamente disponible la información climática generada por sensores en campo.


### 1.2 Problema a Resolver

Los datos de mediciones de campo se acumulaban almacenados en archivos Excel donde se encuentranfragmentados y sin normalización, lo que impede:

- Realizar consultas analíticas complejas de forma eficiente.
- Garantizar la integridad y trazabilidad de los registros.
- Comparar sistemáticamente los dos tratamientos experimentales: **Campo** e **Invernadero**.
- Alimentar futuros modelos predictivos (EDA, ML) con datos limpios y estructurados.

### 1.3 Objetivos del Entregable

| Objetivo | Descripción |
|----------|-------------|
| **Centralización** | Migrar los datos históricos de sensores (2023–2026) a una BD SQL Server relacional |
| **Validación** | Aplicar 8 reglas de negocio agronómicas sobre temperatura, HR, precipitaciones y radiación |
| **Trazabilidad** | Registrar timestamp de carga y flags de auditoría para cada registro |
| **Reproducibilidad** | Encapsular todo el proceso en un notebook Jupyter ejecutable y versionado en GitHub |
| **Escalabilidad** | Diseñar un esquema relacional que soporte futuras tablas de fenología, riego y producción |

---

<a id="2-fuente-de-datos"></a>
## 2. Fuente de Datos

### 2.1 Archivo Origen

| Atributo | Valor |
|----------|-------|
| **Nombre del archivo** | `Dep FIA climas riegos sondas.xlsx` |
| **Hoja activa** | `MedicionesClimaticas` |
| **Dimensiones** | 1.860 filas × 15 columnas |
| **Periodo cubierto** | 2023-11-11 → 2026-05-28 (aprox. 930 días) |
| **Tratamientos** | `Campo` · `Invernadero` (930 registros c/u) |
| **Origen** | Sensores climáticos en unidades productivas de Agrícola HC |
| **Frecuencia** | Diaria (un registro por tratamiento por día) |

### 2.2 Variables Registradas

| # | Variable Excel | Variable SQL | Unidad | Cobertura | Descripción Agronómica |
|---|---------------|-------------|--------|-----------|------------------------|
| 1 | `Fecha` | `fecha` | DATE | 100 % | Fecha de la medición diaria |
| 2 | `Tratamiento` | `tratamiento` | VARCHAR | 100 % | Tipo de unidad productiva |
| 3 | `Temperatura minima (ºC)` | `temp_min` | °C | 100 % | Temperatura mínima diaria |
| 4 | `Temperatura media (ºC)` | `temp_media` | °C | 100 % | Temperatura media diaria |
| 5 | `Temperatura maxima (ºC)` | `temp_max` | °C | 100 % | Temperatura máxima diaria |
| 6 | `Humedad relativa minima (%)` | `hr_min` | % | 100 % | Humedad relativa mínima |
| 7 | `Humedad relativa media (%)` | `hr_media` | % | 100 % | Humedad relativa media |
| 8 | `Humedad relativa maxima (%)` | `hr_max` | % | 100 % | Humedad relativa máxima |
| 9 | `Precipitaciones (mm)` | `precipitaciones_mm` | mm | 45.6 % | Precipitación diaria (solo Campo) |
| 10 | `Piranómetro RG (W/m²)` | `piranometro_rg` | W/m² | 91.2 % | Radiación global incidente |
| 11 | `Rad diaria (MJ/m²*día)` | `rad_diaria_mj` | MJ/m²/d | 91.2 % | Radiación solar diaria |
| 12 | `Integral luz diaria ILD (mol/m²*día)` | `ild_mol` | mol/m²/d | 91.2 % | Integral de luz diaria |
| 13 | `DPV (Kpa)` | `dpv_kpa` | kPa | 91.2 % | Déficit de presión de vapor |
| 14 | `Horas Frío` | `horas_frio` | h | 95.6 % | Horas de frío diarias |
| 15 | `HF acumuladas` | `hf_acumuladas` | h | 91.2 % | Horas de frío acumuladas |

> **Nota:** La baja cobertura de `precipitaciones_mm` (45.6%) se explica porque el tratamiento **Invernadero** no registra precipitaciones externas (protección física). Los valores `NULL` en esta columna son esperados y válidos.

### 2.3 Estadísticas Descriptivas del Dataset Crudo

| Tratamiento | N registros | Período | T° media prom. | HR media prom. | DPV prom. | HF totales | Precip. total |
|-------------|:-----------:|---------|:--------------:|:--------------:|:---------:|:----------:|:-------------:|
| Campo | 930 | 2023-11-11 → 2026-05-28 | 14.62 °C | 83.37 % | 0.361 kPa | 471 h | 274.65 mm |
| Invernadero | 930 | 2023-11-11 → 2026-05-28 | 15.88 °C | 81.15 % | 0.528 kPa | 296 h | 0.00 mm |

---

<a id="3-arquitectura-de-la-solución"></a>
## 3. Arquitectura de la Solución

### 3.1 Diagrama del Pipeline ETL

```
╔══════════════════════════════════════════════════════════════════════════╗
║                    PIPELINE ETL · PROYECTO FIA CHIRIMOYO                ║
╚══════════════════════════════════════════════════════════════════════════╝

  ┌─────────────────────────────┐
  │   SENSORES EN CAMPO         │
  │  Temperatura · HR · Rad.    │
  │  Piranómetro · DPV · HF     │
  └──────────────┬──────────────┘
                 │  Exportación manual
                 ▼
  ┌─────────────────────────────┐
  │   ARCHIVO FUENTE            │
  │  Dep FIA climas...xlsx      │
  │  Hoja: MedicionesClimaticas │
  │  1.860 filas × 15 columnas  │
  └──────────────┬──────────────┘
                 │
  ═══════════════╪═══════════════ FASE E · EXTRACCIÓN ═══════════
                 ▼
  ┌─────────────────────────────┐
  │   [PYTHON · pandas]         │
  │  • pd.read_excel()          │
  │  • Renombrado de columnas   │
  │  • Verificación de headers  │
  │  • Chequeo de columnas      │
  └──────────────┬──────────────┘
                 │
  ═══════════════╪═══════════════ FASE V · VALIDACIÓN ════════════
                 ▼
  ┌─────────────────────────────┐
  │   8 REGLAS DE NEGOCIO       │
  │  R1: Nulos obligatorios     │
  │  R2: Duplicados (fecha+trat)│
  │  R3: Coherencia T° min≤med≤max│
  │  R4: Coherencia HR min≤med≤max│
  │  R5: Rango T° [-5°C, 45°C]  │
  │  R6: Rango HR [0%, 100%]    │
  │  R7: Precip. ≥ 0 mm         │
  │  R8: Radiación/ILD ≥ 0      │
  └──────────────┬──────────────┘
                 │
  ═══════════════╪═══════════════ FASE T · TRANSFORMACIÓN ════════
                 ▼
  ┌─────────────────────────────┐
  │   [PYTHON · pandas]         │
  │  • Estandarizar tratamiento │
  │  • Marcar outliers (flag)   │
  │  • NaN → NULL (pyodbc)      │
  │  • Agregar fecha_carga      │
  │  • Ordenar columnas DDL     │
  └──────────────┬──────────────┘
                 │
  ═══════════════╪═══════════════ FASE L · CARGA ══════════════════
                 ▼
  ┌─────────────────────────────┐
  │   [SQL SERVER · pyodbc]     │
  │  • DDL IF NOT EXISTS        │
  │  • MODO: reemplazar/increm. │
  │  • INSERT por lotes (200)   │
  │  • ROLLBACK por lote        │
  │  • Progress bar en consola  │
  └──────────────┬──────────────┘
                 │
  ═══════════════╪═══════════════ VERIFICACIÓN POST-CARGA ═════════
                 ▼
  ┌─────────────────────────────┐
  │   AUDITORÍA                 │
  │  • COUNT Excel vs SQL Server│
  │  • Distribución tratamiento │
  │  • Revisión flags           │
  │  • Promedios por variable   │
  └─────────────────────────────┘
```

### 3.2 Modo de Carga

El pipeline implementa dos modos de operación controlados por la variable `MODO_CARGA`:

| Modo | Comportamiento | Cuándo usar |
|------|---------------|-------------|
| `"reemplazar"` *(default)* | Vacía la tabla con `DELETE` y recarga el histórico completo | Cuando el Excel entrega el dataset completo actualizado |
| `"incremental"` | Inserta nuevos registros sin borrar los existentes | Cuando el Excel entrega solo filas nuevas (delta) |

> **Decisión de diseño:** Se optó por el modo `"reemplazar"` como default dado que el archivo Excel entregado por los sensores incluye el histórico completo en cada actualización.

---

<a id="4-modelo-relacional"></a>
## 4. Modelo Relacional

### 4.1 Diagrama Entidad-Relación

```
┌──────────────────────────────────────────────────────────────┐
│                        CATALOGO                              │
│                   (Tabla Dimensión)                          │
├─────────────────────┬────────────┬──────────────────────────┤
│ id_tratamiento (PK) │ INT IDENT  │ Clave primaria auto       │
│ sector              │ INT        │ Número de sector          │
│ cuartel             │ VARCHAR    │ Campo / Invernadero       │
│ superficie          │ FLOAT      │ Área total (m²)           │
│ num_plantas         │ INT        │ Total plantas             │
│ plantas_ha          │ FLOAT      │ Plantas por hectárea      │
│ dist_planta         │ FLOAT      │ Distancia entre plantas   │
│ num_got_plan        │ INT        │ Goteros por planta        │
│ caudal_got          │ FLOAT      │ Caudal por gotero (L/h)   │
│ caudal_ha           │ FLOAT      │ Caudal por ha (L/h/ha)    │
│ pp                  │ FLOAT      │ Presión de paso (kPa)     │
│ tratamiento (FK)  ──┼──────────  │ Campo / Invernadero       │
└─────────────────────┴────────────┴──────────────────────────┘
                              │ 1
                              │
                              │ N
┌──────────────────────────────────────────────────────────────┐
│                  MEDICIONESCLIMATICAS                        │
│                    (Tabla de Hechos)                         │
├─────────────────────┬────────────┬──────────────────────────┤
│ id (PK)             │ INT IDENT  │ Clave primaria auto       │
│ fecha               │ DATE       │ NOT NULL                  │
│ tratamiento         │ VARCHAR    │ NOT NULL · FK → Catalogo  │
│ temp_min            │ FLOAT      │ [-5, 45] °C               │
│ temp_media          │ FLOAT      │ Variable de análisis       │
│ temp_max            │ FLOAT      │ [-5, 45] °C               │
│ hr_min              │ FLOAT      │ [0, 100] %                │
│ hr_media            │ FLOAT      │ Variable de análisis       │
│ hr_max              │ FLOAT      │ [0, 100] %                │
│ precipitaciones_mm  │ FLOAT      │ ≥ 0 · nullable            │
│ piranometro_rg      │ FLOAT      │ ≥ 0 W/m² · nullable       │
│ rad_diaria_mj       │ FLOAT      │ ≥ 0 MJ/m²/d · nullable    │
│ ild_mol             │ FLOAT      │ ≥ 0 mol/m²/d · nullable   │
│ dpv_kpa             │ FLOAT      │ ≥ 0 kPa · nullable        │
│ horas_frio          │ FLOAT      │ ≥ 0 h · nullable          │
│ hf_acumuladas       │ FLOAT      │ ≥ 0 h · nullable          │
│ flag_revision       │ BIT        │ 0 = OK · 1 = revisar       │
│ fecha_carga         │ DATETIME   │ Timestamp de ingesta       │
└─────────────────────┴────────────┴──────────────────────────┘

  ✦ Restricción candidata: (fecha, tratamiento) → sin duplicados lógicos
  ✦ Relación: Catalogo.tratamiento (1) → MedicionesClimaticas.tratamiento (N)
```

### 4.2 DDL de la Tabla Principal

```sql
-- ============================================================
-- Tabla: MedicionesClimaticas
-- Base de datos: CH_UCHILE
-- Servidor: LAPTOP-PUELCHE\SQLEXPRESS
-- ============================================================
IF NOT EXISTS (
    SELECT 1 FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_NAME = 'MedicionesClimaticas'
)
BEGIN
    CREATE TABLE MedicionesClimaticas (
        id                 INT IDENTITY(1,1) PRIMARY KEY,
        fecha              DATE          NOT NULL,
        tratamiento        NVARCHAR(50)  NOT NULL,
        temp_min           FLOAT,
        temp_media         FLOAT,
        temp_max           FLOAT,
        hr_min             FLOAT,
        hr_media           FLOAT,
        hr_max             FLOAT,
        precipitaciones_mm FLOAT,
        piranometro_rg     FLOAT,        -- W/m² · nueva variable
        rad_diaria_mj      FLOAT,        -- MJ/m²/día · nueva variable
        ild_mol            FLOAT,        -- mol/m²/día · nueva variable
        dpv_kpa            FLOAT,
        horas_frio         FLOAT,
        hf_acumuladas      FLOAT,
        flag_revision      BIT           NOT NULL DEFAULT 0,
        fecha_carga        DATETIME      NOT NULL
    );
END
-- Agrega columnas nuevas si la tabla ya existía en versión anterior
IF NOT EXISTS (SELECT 1 FROM sys.columns 
               WHERE Name = 'piranometro_rg' 
               AND Object_ID = OBJECT_ID('MedicionesClimaticas'))
    ALTER TABLE MedicionesClimaticas ADD piranometro_rg FLOAT;
```

---

<a id="5-infraestructura-tecnológica"></a>
## 5. Infraestructura Tecnológica

### 5.1 Stack Tecnológico

| Componente | Versión | Rol en el Pipeline |
|-----------|---------|--------------------|
| **Python** | 3.13.9 | Lenguaje principal del ETL |
| **Jupyter Notebook** | ≥ 7.0 | Entorno de desarrollo interactivo |
| **pandas** | ≥ 2.0 | Lectura Excel, manipulación DataFrames, validación |
| **numpy** | ≥ 1.26 | Detección de NaN para conversión a NULL |
| **pyodbc** | ≥ 4.0 | Driver de conexión a SQL Server |
| **openpyxl** | ≥ 3.0 | Motor de lectura de archivos `.xlsx` |
| **SQL Server Express** | 16.0.1180 | Motor de base de datos relacional |
| **ODBC Driver** | 17 for SQL Server | Capa de comunicación Python ↔ SQL Server |
| **GitHub** | — | Control de versiones del proyecto |

### 5.2 Configuración de Conexión

```python
# ── Parámetros de conexión ─────────────────────────────────────────────────
conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=LAPTOP-PUELCHE\\SQLEXPRESS;"
    "DATABASE=CH_UCHILE;"
    "UID=sa;"
    "PWD=<contraseña_segura>;"
    "TrustServerCertificate=yes;"
)
```

| Parámetro | Valor |
|-----------|-------|
| Servidor | `LAPTOP-PUELCHE\SQLEXPRESS` |
| Base de Datos | `CH_UCHILE` |
| Autenticación | SQL Server (usuario `sa`) |
| Certificado | `TrustServerCertificate=yes` |

### 5.3 Repositorio GitHub

```
Marce-717/Chirimoyo/
├── ETL_Agricola_HC.ipynb          ← Pipeline ETL principal
├── Entregable_FIA.ipynb           ← Informe técnico de entrega
├── Dep_FIA_climas_riegos.xlsx     ← Datos fuente (sensores)
└── README.md
```

---

<a id="6-pipeline-etl--detalle-técnico"></a>
## 6. Pipeline ETL · Detalle Técnico

El pipeline se articula en 7 celdas ejecutables dentro del notebook `ETL_Agricola_HC.ipynb`, cada una con una función delimitada.

### Celda 0 · Conexión a la Base de Datos

**Propósito:** Establecer y verificar la conexión al servidor SQL Server local antes de iniciar el proceso de carga. Si la conexión falla, el pipeline detiene la ejecución con un mensaje descriptivo.

In [2]:
import pyodbc
import pandas as pd
import numpy as np
import math
from datetime import datetime

conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=LAPTOP-PUELCHE\\SQLEXPRESS;"
    "DATABASE=CH_UCHILE;"
    "UID=sa;"
    "PWD=<contraseña_segura>;"
    "TrustServerCertificate=yes;"
)

try:
    cnxn   = pyodbc.connect(conn_str)
    cursor = cnxn.cursor()
    print("✅ Conexión exitosa a CH_UCHILE")
except pyodbc.Error as e:
    print(f"❌ Error de conexión: {e}")

❌ Error de conexión: ('28000', "[28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Login failed for user 'sa'. (18456) (SQLDriverConnect); [28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Login failed for user 'sa'. (18456)")


### Celda 1 · EXTRACCIÓN: Carga del Excel y Estandarización de Nombres

**Propósito:** Leer el archivo fuente con `pandas.read_excel()`, aplicar inmediatamente el **diccionario de renombrado** que traduce los encabezados Excel con caracteres especiales y espacios a nombres SQL-friendly, y verificar que todas las columnas esperadas estén presentes.

**Criterio de diseño:** El renombrado se realiza en la Fase E (no en la T) para que las reglas de validación de la Celda 2 ya trabajen con los nombres definitivos, reduciendo el riesgo de inconsistencias.

In [3]:
EXCEL_PATH  = r"C:/ruta/a/Dep FIA climas riegos sondas.xlsx"
SHEET_NAME  = "MedicionesClimaticas"
TABLE_DEST  = "MedicionesClimaticas"

# Diccionario Excel headers → nombres SQL-friendly
RENOMBRE_COLUMNAS = {
    "Fecha"                                : "fecha",
    "Tratamiento"                          : "tratamiento",
    "Temperatura minima (ºC)"              : "temp_min",
    "Temperatura media (ºC)"               : "temp_media",
    "Temperatura maxima (ºC)"              : "temp_max",
    "Humedad relativa minima (%)"          : "hr_min",
    "Humedad relativa media (%)"           : "hr_media",
    "Humedad relativa maxima (%)"          : "hr_max",
    "Precipitaciones (mm)"                 : "precipitaciones_mm",
    "Piranómetro RG (W/m2)"                : "piranometro_rg",
    "Rad diaria (MJ/m2*dia)"               : "rad_diaria_mj",
    "Integral luz diaria ILD (mol/m2*día)" : "ild_mol",
    "DPV (Kpa)"                            : "dpv_kpa",
    "Horas Frío"                           : "horas_frio",
    "HF acumuladas"                        : "hf_acumuladas",
}

df_raw = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)
df_raw.rename(columns=RENOMBRE_COLUMNAS, inplace=True)

# Verificación de columnas esperadas
faltantes = set(RENOMBRE_COLUMNAS.values()) - set(df_raw.columns)
if faltantes:
    print(f"⚠️  Columnas faltantes: {faltantes}")

print(f"Filas cargadas : {len(df_raw):>6,}")
print(f"Columnas       : {len(df_raw.columns):>6}")
print(f"Período        : {df_raw['fecha'].min().date()}  →  {df_raw['fecha'].max().date()}")
df_raw.head()

FileNotFoundError: [Errno 2] No such file or directory: 'C:/ruta/a/Dep FIA climas riegos sondas.xlsx'

### Celda 2 · VALIDACIÓN: 8 Reglas de Negocio

**Propósito:** Auditar la calidad del dataset crudo antes de cualquier transformación. El resultado es un **reporte de calidad** que lista los problemas encontrados por regla, columna y número de filas afectadas.

Las reglas se diseñaron con criterio agronómico específico para el cultivo de chirimoya en las condiciones climáticas del área productiva de Agrícola HC.

In [ ]:
errores = []

# R1: Nulos en columnas obligatorias (temp, HR, fecha, tratamiento)
OBLIGATORIAS = ["fecha","tratamiento","temp_min","temp_media","temp_max",
                "hr_min","hr_media","hr_max"]
for col in OBLIGATORIAS:
    n = df_raw[col].isnull().sum()
    if n > 0:
        errores.append({"regla":"Nulo_obligatorio","columna":col,"n_filas":n})

# R2: Duplicados exactos (fecha + tratamiento)
mask_dup = df_raw.duplicated(subset=["fecha","tratamiento"], keep=False)
if mask_dup.sum() > 0:
    errores.append({"regla":"Duplicado","columna":"fecha+tratamiento","n_filas":int(mask_dup.sum())})

# R3: Coherencia temperatura (min ≤ media ≤ max)
mask_t = (df_raw["temp_min"] > df_raw["temp_media"]) | (df_raw["temp_media"] > df_raw["temp_max"])
if mask_t.sum() > 0:
    errores.append({"regla":"Coherencia_temp","columna":"temp_min/media/max","n_filas":int(mask_t.sum())})

# R4: Coherencia HR (min ≤ media ≤ max)
mask_h = (df_raw["hr_min"] > df_raw["hr_media"]) | (df_raw["hr_media"] > df_raw["hr_max"])
if mask_h.sum() > 0:
    errores.append({"regla":"Coherencia_HR","columna":"hr_min/media/max","n_filas":int(mask_h.sum())})

# R5: Rango temperatura válida para chirimoyos [-5°C, 45°C]
mask_tr = (df_raw["temp_min"] < -5) | (df_raw["temp_max"] > 45)
if mask_tr.sum() > 0:
    errores.append({"regla":"Outlier_temp","columna":"temp_min/temp_max","n_filas":int(mask_tr.sum())})

# R6: Rango HR válido [0%, 100%]
mask_hr = (df_raw["hr_min"] < 0) | (df_raw["hr_max"] > 100)
if mask_hr.sum() > 0:
    errores.append({"regla":"Outlier_HR","columna":"hr_min/hr_max","n_filas":int(mask_hr.sum())})

# R7: Precipitaciones no negativas
mask_pp = df_raw["precipitaciones_mm"].notna() & (df_raw["precipitaciones_mm"] < 0)
if mask_pp.sum() > 0:
    errores.append({"regla":"Negativo_precip","columna":"precipitaciones_mm","n_filas":int(mask_pp.sum())})

# R8: Variables de radiación/luz no negativas
for col in ["piranometro_rg","rad_diaria_mj","ild_mol"]:
    mask_r = df_raw[col].notna() & (df_raw[col] < 0)
    if mask_r.sum() > 0:
        errores.append({"regla":"Negativo_radiacion","columna":col,"n_filas":int(mask_r.sum())})

# ── Reporte ────────────────────────────────────────────────────────────────
print("=" * 60)
print(" REPORTE DE CALIDAD DE DATOS")
print("=" * 60)
print(f" Total filas analizadas : {len(df_raw):,}")
print(f" Problemas detectados   : {len(errores)}")
print("-" * 60)
if errores:
    df_err = pd.DataFrame(errores)
    print(df_err.to_string(index=False))
    print("\n⚠️  Ver celda de LIMPIEZA antes de ingestar.")
else:
    print(" ✅ Sin errores críticos. Dataset listo para ingesta.")
print("=" * 60)

### Celda 3 · TRANSFORMACIÓN: Limpieza y Auditoría

**Propósito:** Aplicar las correcciones detectadas en la Celda 2 y enriquecer el dataset con columnas de auditoría necesarias para la trazabilidad en SQL Server.

**Criterio de tratamiento de outliers:** Los registros con temperaturas fuera del rango fisiológico del chirimoyo **no se eliminan** — se preservan con `flag_revision = True` para análisis posterior. Esto permite mantener la integridad del histórico y auditar los datos con el encargado de campo.

In [ ]:
df = df_raw.copy()

# 3.1 Estandarizar texto en la columna tratamiento
df["tratamiento"] = df["tratamiento"].str.strip().str.capitalize()

# 3.2 Marcar filas con outlier de temperatura (no se eliminan)
df["flag_revision"] = False
mask_outlier = (df["temp_min"] < -5) | (df["temp_max"] > 45)
df.loc[mask_outlier, "flag_revision"] = True

# 3.3 Agregar columna de trazabilidad (timestamp de carga)
df["fecha_carga"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# 3.4 Resumen post-transformación
print(f"Filas listas para ingestar : {len(df):,}")
print(f"Filas con flag_revision=1  : {df['flag_revision'].sum()}")
print()
print("Columnas finales:")
for col, dtype in df.dtypes.items():
    nulos = df[col].isnull().sum()
    print(f"  {col:<25} {str(dtype):<15} nulos: {nulos}")

### Celda 4 · DDL: Creación de Tabla (primera ejecución)

**Propósito:** Crear la tabla `MedicionesClimaticas` en SQL Server si no existe. El script es **idempotente** — puede ejecutarse múltiples veces sin error. Además, detecta tablas creadas en versiones anteriores del esquema (sin las columnas de radiación) y las actualiza con `ALTER TABLE`.

In [ ]:
# Ver Celda 4 en ETL_Agricola_HC.ipynb
# Ejecutar solo en la primera inicialización de la BD
# El DDL completo incluye IF NOT EXISTS + ALTER TABLE para columnas nuevas
print("▶ Ejecutar ETL_Agricola_HC.ipynb › Celda 4 para el DDL completo")

### Celda 5 · CARGA: Ingesta por Lotes con Control de Errores

**Propósito:** Insertar los datos transformados en SQL Server mediante `executemany()` con lotes de 200 filas, barra de progreso en consola y manejo de errores a nivel de lote (rollback por lote fallido, continúa con el siguiente).

**Parámetros de ingesta:**

| Parámetro | Valor | Descripción |
|-----------|-------|-------------|
| `BATCH_SIZE` | 200 | Filas por lote de inserción |
| `MODO_CARGA` | `"reemplazar"` | Vacía la tabla y recarga el histórico completo |
| NaN → NULL | `math.isnan()` | Conversión de valores NaN de pandas a NULL de SQL |
| Rollback | Por lote | Si un lote falla, se hace rollback solo de ese lote |

In [ ]:
# Ver implementación completa en ETL_Agricola_HC.ipynb › Celda 5
# Resumen del flujo de ingesta:

# if MODO_CARGA == "reemplazar":
#     cursor.execute(f"DELETE FROM {TABLE_DEST}")
#     cnxn.commit()
#
# COLS_SQL = ["fecha","tratamiento","temp_min",...,"flag_revision","fecha_carga"]
# sql_insert = f"INSERT INTO {TABLE_DEST} ({cols}) VALUES ({placeholders})"
#
# for lote_i in range(n_lotes):          # lotes de 200 filas
#     try:
#         cursor.executemany(sql_insert, datos)
#         cnxn.commit()
#     except pyodbc.Error:
#         cnxn.rollback()                # rollback del lote fallido

print("▶ Ejecutar ETL_Agricola_HC.ipynb › Celda 5 para la ingesta completa")

---

<a id="7-calidad-de-datos"></a>
## 7. Calidad de Datos

### 7.1 Resumen de Reglas de Negocio Aplicadas

| # | Regla | Variable(s) | Criterio Agronómico | Acción |
|---|-------|------------|---------------------|---------|
| R1 | Nulos obligatorios | `fecha`, `tratamiento`, `temp_*`, `hr_*` | Sin datos base no hay registro válido | Detiene la ingesta |
| R2 | Duplicados | `fecha + tratamiento` | Un sensor por tratamiento por día | Reporta, no inserta |
| R3 | Coherencia temperatura | `temp_min ≤ temp_media ≤ temp_max` | Ley física: mín ≤ media ≤ máx | Flag de revisión |
| R4 | Coherencia HR | `hr_min ≤ hr_media ≤ hr_max` | Ídem para humedad relativa | Flag de revisión |
| R5 | Rango temperatura | `−5°C ≤ temp ≤ 45°C` | Rango fisiológico chirimoyo | Flag de revisión |
| R6 | Rango HR | `0% ≤ HR ≤ 100%` | Límites físicos del sensor | Flag de revisión |
| R7 | Precipitaciones | `precip ≥ 0 mm` | Magnitud física no negativa | Flag de revisión |
| R8 | Radiación / ILD | `Rad ≥ 0, ILD ≥ 0` | Energía solar siempre positiva | Flag de revisión |

### 7.2 Hallazgos del Dataset Analizado

| Regla | Estado | Detalle |
|-------|--------|----------|
| R1 – Nulos obligatorios | ✅ Sin errores | Temperatura y HR al 100% de cobertura |
| R2 – Duplicados | ✅ Sin errores | 930 filas únicas por tratamiento |
| R3 – Coherencia temp | ✅ Sin errores | `min ≤ media ≤ max` en todos los registros |
| R4 – Coherencia HR | ✅ Sin errores | Coherencia de humedad confirmada |
| R5 – Rango temperatura | ⚠️ 1 registro | 2026-04-18 · Campo: `temp_max = 95.38°C` — probable error sensor |
| R6 – Rango HR | ✅ Sin errores | Todos los valores dentro de [0%, 100%] |
| R7 – Precipitaciones | ✅ Sin errores | No hay valores negativos |
| R8 – Radiación/ILD | ✅ Sin errores | Piranómetro y DPV sin negativos |

### 7.3 Registro Anómalo Detectado

| Campo | Valor |
|-------|-------|
| `fecha` | 2026-04-18 |
| `tratamiento` | Campo |
| `temp_min` | 11.71 °C |
| `temp_media` | **53.54 °C** ← anómalo |
| `temp_max` | **95.38 °C** ← fuera de rango |
| `flag_revision` | **1 (requiere revisión)** |

> **Interpretación:** La temperatura máxima de 95.38°C es físicamente imposible en condiciones de campo para chirimoyos. Se presume error de lectura del sensor o fallo temporal de transmisión de datos. El registro **se preserva en la BD** con `flag_revision = 1` para verificación con el equipo de campo.

---

<a id="8-resultados-de-la-ingesta"></a>
## 8. Resultados de la Ingesta

### 8.1 Métricas de Carga

| Métrica | Valor |
|---------|-------|
| Filas en Excel | 1.860 |
| Filas cargadas en SQL Server | 1.860 |
| Coincidencia | ✅ 100% |
| Lotes procesados | 10 (de 200 filas c/u) |
| Lotes fallidos | 0 |
| Filas con `flag_revision = 1` | 1 |
| Modo de carga utilizado | `reemplazar` |
| Tiempo estimado de carga | < 30 segundos |

### 8.2 Distribución por Tratamiento (Post-Ingesta)

```sql
SELECT 
    tratamiento,
    COUNT(*)                       AS n_registros,
    MIN(fecha)                     AS fecha_inicio,
    MAX(fecha)                     AS fecha_fin,
    ROUND(AVG(temp_media), 2)      AS temp_media_prom,
    ROUND(AVG(hr_media), 2)        AS hr_media_prom,
    ROUND(AVG(dpv_kpa), 3)         AS dpv_prom,
    ROUND(AVG(piranometro_rg), 1)  AS piranometro_prom,
    ROUND(SUM(horas_frio), 0)      AS hf_totales
FROM MedicionesClimaticas
GROUP BY tratamiento
ORDER BY tratamiento;
```

| Tratamiento | N registros | Período | T° media | HR media | DPV | Piranómetro | HF totales |
|-------------|:-----------:|---------|:--------:|:--------:|:---:|:-----------:|:----------:|
| Campo | 930 | 2023-11-11 → 2026-05-28 | 14.62 °C | 83.37 % | 0.361 kPa | 329.3 W/m² | 471 h |
| Invernadero | 930 | 2023-11-11 → 2026-05-28 | 15.88 °C | 81.15 % | 0.528 kPa | 329.3 W/m² | 296 h |

### 8.3 Análisis de Cobertura de Datos en la BD

```sql
SELECT
    COUNT(*)                                                     AS total,
    ROUND(100.0 * COUNT(precipitaciones_mm) / COUNT(*), 1)      AS pct_precip,
    ROUND(100.0 * COUNT(piranometro_rg) / COUNT(*), 1)          AS pct_rad,
    ROUND(100.0 * COUNT(dpv_kpa) / COUNT(*), 1)                 AS pct_dpv,
    ROUND(100.0 * COUNT(horas_frio) / COUNT(*), 1)              AS pct_hf
FROM MedicionesClimaticas;
```

| Variable | Cobertura | Observación |
|----------|:---------:|-------------|
| Temperatura (min/media/max) | 100.0 % | Completa |
| Humedad relativa (min/media/max) | 100.0 % | Completa |
| Precipitaciones | 45.6 % | Invernadero = NULL por diseño |
| Piranómetro / Radiación / ILD | 91.2 % | Primeras semanas sin sensor |
| DPV | 91.2 % | Idem piranómetro |
| Horas de frío | 95.6 % | Vacíos inicio de temporada |
| HF acumuladas | 91.2 % | Calculadas desde inicio temporada |

---

<a id="9-archivos-entregables"></a>
## 9. Archivos Entregables

### 9.1 Inventario de Archivos

| Archivo | Tipo | Descripción | Versión |
|---------|------|-------------|---------|
| `ETL_Agricola_HC.ipynb` | Jupyter Notebook | Pipeline ETL ejecutable completo (7 celdas) | v2.0 |
| `Entregable_FIA.ipynb` | Jupyter Notebook | Este informe técnico de diseño y resultados | v2.0 |
| `Informe_ETL_AgricolaHC_FIA.ipynb` | Jupyter Notebook | Versión profesional consolidada del informe | v2.0 |
| `Dep_FIA_climas_riegos_sondas.xlsx` | Excel | Datos fuente de sensores (1.860 registros) | Entrega 3 |

### 9.2 Repositorio GitHub

```
https://github.com/Marce-717/Chirimoyo
```

Todos los archivos del proyecto se encuentran versionados en el repositorio público de GitHub. Cada commit documenta los cambios realizados en el pipeline ETL.

### 9.3 Próximos Pasos Sugeridos

| Prioridad | Tarea | Descripción |
|:---------:|-------|-------------|
| Alta | Verificar registro anómalo | Consultar al equipo de campo sobre la medición del 2026-04-18 (`temp_max = 95.38°C`) |
| Alta | Poblar tabla `Catalogo` | Ingresar los datos de riego y estructura del sistema de goteo por sector |
| Media | EDA en Python | Análisis exploratorio de temperatura y horas de frío por temporada |
| Media | Visualizaciones SQL | Dashboards de evolución temporal de variables climáticas por tratamiento |
| Baja | Automatización | Configurar tarea programada para actualización periódica del Excel → BD |
| Baja | Modelo predictivo | Correlación entre acumulación de horas de frío y fenología (floración/cuaja) |

---

<div style="background: #f0f7f0; border-left: 4px solid #2d5a27; padding: 20px; border-radius: 0 8px 8px 0; margin-top: 30px;">
<h3 style="margin-top: 0; color: #2d5a27;">📌 Notas de Versión</h3>

<p><strong>v2.0 · Julio 2026</strong></p>
<ul>
<li>Incorporación de 3 nuevas variables: <code>piranometro_rg</code>, <code>rad_diaria_mj</code>, <code>ild_mol</code></li>
<li>Regla de negocio R8 agregada para validación de radiación/ILD</li>
<li>Migración de base de datos: <code>FIA</code> → <code>CH_UCHILE</code></li>
<li>Modo de carga <code>"reemplazar"</code> implementado para evitar duplicados en recargas</li>
<li>Informe técnico consolidado en formato notebook</li>
</ul>

<p><strong>v1.0 · Junio 2026</strong></p>
<ul>
<li>Versión inicial con 7 reglas de validación</li>
<li>Ingesta de 1.860 registros históricos (2023–2026)</li>
<li>Esquema relacional con tablas <code>MedicionesClimaticas</code> y <code>Catalogo</code></li>
</ul>
</div>

---

<div style="text-align: center; padding: 20px; color: #666; font-size: 0.9em; border-top: 1px solid #ddd; margin-top: 30px;">
<strong>Proyecto FIA · Chirimoyo · Agrícola HC</strong><br>
Universidad de Chile · Facultad de Ciencias Agronómicas<br>
Repositorio: <a href="https://github.com/Marce-717/Chirimoyo">github.com/Marce-717/Chirimoyo</a><br>
<em>Documento generado: Julio 2026</em>
</div>